# Original Client Return Report - Cleaned Notebook

This is a cleaned version of the original Colab notebook. It keeps the main workflow: read two Excel files per client, combine the data, calculate report values, and write the results into a template workbook.

Removed from the original:

- Google Drive mounting and hardcoded Colab paths
- package install commands
- scratch and experiment cells
- automatic execution at the end


## 1. Imports

In [ ]:
from __future__ import annotations

import datetime
import re
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from pytz import timezone


## 2. Configuration

Change these paths before running the notebook.

In [ ]:
BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"
TEMPLATE_FILE = BASE_DIR / "Templet.xlsx"
OUTPUT_FILE = OUTPUT_DIR / "client_report.xlsx"

TZ_AU = timezone("Australia/Sydney")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Base folder:", BASE_DIR)
print("Input folder:", INPUT_DIR)
print("Output file:", OUTPUT_FILE)


## 3. Optional CSV Conversion

The original notebook converted CSV files into XLSX files before processing.

In [ ]:
def convert_csv_files_to_xlsx(input_dir: Path) -> None:
    import csv
    from xlsxwriter.workbook import Workbook

    for csv_file in input_dir.glob("*.csv"):
        xlsx_file = csv_file.with_suffix(".xlsx")
        workbook = Workbook(str(xlsx_file))
        worksheet = workbook.add_worksheet()

        with csv_file.open("rt", encoding="utf-8") as handle:
            reader = csv.reader(handle)
            for row_number, row in enumerate(reader):
                for col_number, value in enumerate(row):
                    worksheet.write(row_number, col_number, value)

        workbook.close()
        csv_file.unlink()
        print(f"Converted {csv_file.name} -> {xlsx_file.name}")


## 4. Match Client Files

This keeps the original filename-parsing idea, but uses a local input folder instead of Google Drive.

In [ ]:
def build_client_file_association(input_dir: Path) -> dict[str, list[str]]:
    filenames = [path.name for path in input_dir.iterdir() if path.is_file()]
    association: dict[str, list[str]] = {}

    for filename in filenames:
        parts = re.split("_", filename)
        if len(parts) > 1:
            sheet_name = parts[0].split(" ")[0]
            client_id = parts[1]
            association[client_id] = [sheet_name, str(input_dir / filename)]

    for filename in filenames:
        parts = re.split("-", filename)
        if len(parts) > 1:
            client_id = parts[1]
            if client_id in association:
                association[client_id].append(str(input_dir / filename))

    return association


assoDict = build_client_file_association(INPUT_DIR) if INPUT_DIR.exists() else {}
for client_id in sorted(assoDict):
    print(client_id, assoDict[client_id])


## 5. Data Cleaning Helpers

In [ ]:
def check_df_line_empty(row_index: int, df: pd.DataFrame) -> bool:
    return df.iloc[row_index].isna().all()


def check_df_col_empty(col_index: int, df: pd.DataFrame) -> bool:
    return df.iloc[:, col_index].isna().all()


def pop_empty_cols(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna(axis=1, how="all")


def drop_trailing_nan(df: pd.DataFrame) -> pd.DataFrame:
    df = df.reset_index(drop=True)
    for i in range(len(df) - 1):
        if pd.isna(df.loc[i, "Asset"]) and pd.isna(df.loc[i + 1, "Asset"]):
            return df.iloc[:i].copy()
    return df


## 6. Read the Two Client Excel Files

In [ ]:
def load_asset_performance(path: str | Path) -> pd.DataFrame:
    df_ap = pd.read_excel(path)
    df_ap = pop_empty_cols(df_ap)

    df_ap_0 = df_ap.iloc[6:, :].copy()
    df_ap_0.columns = df_ap_0.iloc[0, :]
    df_ap_0 = df_ap_0.iloc[2:].reset_index(drop=True)

    df_ap_1 = df_ap_0[[
        "Asset",
        "Code",
        "Opening Value",
        "Closing\n value",
        "Return for\nPeriod % **",
    ]].reset_index(drop=True)

    return drop_trailing_nan(df_ap_1)


def generate_df_mp_0(df: pd.DataFrame) -> pd.DataFrame:
    in_table = False
    rows = []
    columns = None

    for i in range(len(df)):
        row_values = list(df.iloc[i, :].values)
        first_value = row_values[0]

        if in_table:
            rows.append(row_values)

        if first_value == "Asset":
            columns = row_values
            in_table = True
        elif first_value == "Total Portfolio":
            break

    return pd.DataFrame(rows, columns=columns)


def load_myportfolio(path: str | Path) -> pd.DataFrame:
    df_mp = pd.read_excel(path)
    df_mp_0 = generate_df_mp_0(df_mp)
    return df_mp_0[["Code", "Cost $", "Value %"]].copy()


## 7. Combine and Shape Report Data

In [ ]:
def manual_join(df_ap_1: pd.DataFrame, df_mp_1: pd.DataFrame) -> pd.DataFrame:
    df_out = df_ap_1.copy()
    df_out.insert(2, "Adjusted cost $", np.nan)
    df_out.insert(5, "Total Return %", np.nan)
    df_out.insert(6, "Unrealised capital gain $", np.nan)
    df_out.insert(7, "Value %", np.nan)

    df_out = df_out.rename({
        "Opening Value": "Market value $ (last week)",
        "Closing\n value": "Market value $",
        "Return for\nPeriod % **": "Weekly Return %",
    }, axis=1)

    for i in range(len(df_out)):
        for j in range(len(df_mp_1)):
            if df_out.loc[i, "Code"] == df_mp_1.loc[j, "Code"]:
                df_out.loc[i, "Adjusted cost $"] = df_mp_1.loc[j, "Cost $"]
                df_out.loc[i, "Value %"] = float("%.2f" % float(df_mp_1.loc[j, "Value %"]))

    return df_out


def find_cash(df_out: pd.DataFrame) -> dict[str, object]:
    cash = {}
    foreign_currency_start = None
    foreign_currency_end = None
    finding_foreign_currency_total = False

    for i in range(len(df_out)):
        asset = df_out.loc[i, "Asset"]
        if asset == "Cash Account":
            cash["Cash Account"] = df_out.loc[i, "Market value $"]
        elif asset == "Foreign Currency":
            foreign_currency_start = i
            finding_foreign_currency_total = True
        elif finding_foreign_currency_total and asset == "Sub Total":
            foreign_currency_end = i
            finding_foreign_currency_total = False

    if foreign_currency_start is not None and foreign_currency_end is not None:
        for j in range(foreign_currency_start + 1, foreign_currency_end):
            cash[df_out.loc[j, "Asset"]] = df_out.loc[j, "Market value $"]

    return cash


In [ ]:
def fill_row_header(df: pd.DataFrame, row_index: int) -> None:
    for j in range(len(df.loc[row_index]) - 1):
        df.iloc[row_index, j + 1] = df.columns[j + 1]


def find_block_start_end(df: pd.DataFrame) -> tuple[list[int], list[int]]:
    block_start = []
    block_end = []
    finding_subtotal = False

    for i in range(len(df)):
        asset = df.loc[i, "Asset"]
        if asset in ["ASX - Australian Shares", "International Shares - NASDAQ"]:
            fill_row_header(df, i)
            block_start.append(i)
            finding_subtotal = True
        elif asset == "Total Portfolio Value":
            block_start.append(i)

        if finding_subtotal and asset == "Sub Total":
            block_end.append(i)
            finding_subtotal = False

    return block_start, block_end


def add_row_headers(df: pd.DataFrame, block_start: list[int], block_end: list[int]) -> pd.DataFrame:
    frames = []
    for i, start in enumerate(block_start):
        if i == len(block_start) - 1:
            frames.append(df.iloc[start:start + 1])
        else:
            end = block_end[i]
            frames.append(df.iloc[start:end + 1])
            if i + 2 < len(block_start):
                frames.append(df.iloc[end + 1:end + 2])
    return pd.concat(frames, ignore_index=True) if frames else df.copy()


## 8. Template Workbook Helpers

In [ ]:
def get_investments_and_BWM_row_num(target_sheet) -> tuple[list[int], list[int]]:
    investments = []
    rows_BWM_block = []
    row_num = 1
    found_investment_block = False

    while True:
        value = target_sheet.cell(row=row_num, column=5).value
        if value is not None:
            found_investment_block = True
            try:
                investments.append(int(value))
                rows_BWM_block.append(row_num)
            except (TypeError, ValueError):
                pass
        elif found_investment_block:
            break
        row_num += 1

    return investments, rows_BWM_block


def get_date_and_time_diff(target_sheet, row_num: int) -> tuple[str, int]:
    time_now = datetime.datetime.today().astimezone(TZ_AU)
    date_today = datetime.datetime.strftime(time_now, "%d/%m/%Y")
    start_date = target_sheet.cell(row=row_num, column=2).value

    if hasattr(start_date, "astimezone"):
        start_date = start_date.astimezone(TZ_AU)

    period_length = time_now - start_date
    return date_today, period_length.days


def write_date_and_diff(target_sheet, rows_BWM_block: list[int], investments: list[int]) -> int:
    total_adjusted_days = 0
    for i, row_num in enumerate(rows_BWM_block):
        date_today, days = get_date_and_time_diff(target_sheet, row_num)
        target_sheet.cell(row=row_num, column=3).value = date_today
        total_adjusted_days += investments[i] * days / sum(investments)

    target_sheet.cell(row=rows_BWM_block[-1] + 1, column=2).value = int(total_adjusted_days)
    target_sheet.cell(row=rows_BWM_block[-1] + 1, column=3).value = int(total_adjusted_days) // 7
    return int(total_adjusted_days)


def write_cash(target_sheet, rows_BWM_block: list[int], cash: dict[str, object]) -> None:
    row_num = rows_BWM_block[-1] + 2
    for key, value in cash.items():
        target_sheet.cell(row=row_num, column=1).value = key
        target_sheet.cell(row=row_num, column=2).value = value
        row_num += 1


def write_annualized_rate(target_sheet, row_annual: int, total_adjusted_days: int, total_portfolio_return: float) -> None:
    target_sheet.cell(row=row_annual + 1, column=6).value = "Annualized rate"
    target_sheet.cell(row=row_annual + 1, column=7).value = "%.2f" % (total_portfolio_return / total_adjusted_days * 365) + "%"


## 9. Final Calculations

In [ ]:
def calculate_df(df_out_1: pd.DataFrame, total_investments: float) -> tuple[pd.DataFrame, float, pd.Series]:
    block_start, block_end = find_block_start_end(df_out_1)
    total_portfolio_return = 0.0
    total_portfolio_market_value = pd.Series(dtype="float64")

    for i in range(len(df_out_1)):
        try:
            df_out_1.loc[i, "Unrealised capital gain $"] = float("%.2f" % (
                float(df_out_1.loc[i, "Market value $"]) - float(df_out_1.loc[i, "Adjusted cost $"])
            ))
        except (TypeError, ValueError):
            pass

        for j, end in enumerate(block_end):
            if i == end:
                start = block_start[j]
                sum_cols = [
                    "Adjusted cost $",
                    "Market value $ (last week)",
                    "Market value $",
                    "Unrealised capital gain $",
                ]
                df_out_1.loc[i, sum_cols] = df_out_1.loc[start + 1:end - 1, sum_cols].sum()
            elif block_start and i == block_start[-1]:
                df_out_1.loc[i, ["Adjusted cost $"]] = df_out_1.loc[block_end, ["Adjusted cost $"]].sum()
                total_portfolio_return = (float(df_out_1.loc[i, ["Market value $"]]) / float(total_investments) - 1.0) * 100
                df_out_1.loc[i, ["Total Return %"]] = "%.2f" % total_portfolio_return + "%"
                total_portfolio_market_value = df_out_1.loc[i, ["Market value $"]]

        if i not in block_end and block_start and i != block_start[-1]:
            try:
                df_out_1.loc[i, "Total Return %"] = "%.2f" % (
                    float(df_out_1.loc[i, "Unrealised capital gain $"]) / float(df_out_1.loc[i, "Adjusted cost $"]) * 100
                )
                df_out_1.loc[i, "Weekly Return %"] = "%.2f" % ((
                    float(df_out_1.loc[i, "Market value $"]) - float(df_out_1.loc[i, "Market value $ (last week)"])
                ) / float(df_out_1.loc[i, "Market value $ (last week)"]) * 100)
            except (TypeError, ValueError, ZeroDivisionError):
                pass

    return df_out_1.replace("nan", ""), total_portfolio_return, total_portfolio_market_value


## 10. Process One Client

In [ ]:
def write_sheet(association: dict[str, list[str]], client_id: str, workbook, writer) -> np.ndarray:
    sheet_name = association[client_id][0]
    myportfolio_file = association[client_id][1]
    asset_performance_file = association[client_id][2]

    df_ap_1 = load_asset_performance(asset_performance_file)
    df_mp_1 = load_myportfolio(myportfolio_file)

    df_out = manual_join(df_ap_1, df_mp_1)
    cash = find_cash(df_out)
    block_start, block_end = find_block_start_end(df_out)
    df_out_1 = add_row_headers(df_out.copy(), block_start, block_end)

    target_sheet = workbook[sheet_name]
    investments, rows_BWM_block = get_investments_and_BWM_row_num(target_sheet)
    write_cash(target_sheet, rows_BWM_block, cash)
    total_adjusted_days = write_date_and_diff(target_sheet, rows_BWM_block, investments)

    row_df_start = rows_BWM_block[-1] + 1 + len(cash)
    df_out_1, total_portfolio_return, total_portfolio_market_value = calculate_df(df_out_1, sum(investments))
    df_out_1.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=row_df_start, startcol=0)

    row_annual = row_df_start + len(df_out_1)
    write_annualized_rate(target_sheet, row_annual, total_adjusted_days, total_portfolio_return)

    return total_portfolio_market_value.values


## 11. Main Function

Run this only after checking the configured folders and template file.

In [ ]:
def main() -> None:
    association = build_client_file_association(INPUT_DIR)
    workbook = load_workbook(filename=TEMPLATE_FILE)
    writer = pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl")
    writer.book = workbook
    writer.sheets = {worksheet.title: worksheet for worksheet in workbook.worksheets}

    total_fund = 0
    for client_id in sorted(association):
        client_fund = write_sheet(association, client_id, workbook, writer)
        total_fund += client_fund
        print(client_id, client_fund)

    workbook["001"].cell(row=2, column=2).value = total_fund[0]
    writer.save()
    writer.close()
    print("Saved report to", OUTPUT_FILE)


# Uncomment this line when ready to run the full process.
# main()
